# Relationships Between Two Variables

### Does spending more move the result the business cares about?

A marketing manager asks: *if we spend £1,000 more on advertising each month, how many more customers should we expect, and how far can we trust that answer?* This notebook answers that question in Python and checks the answer against the same analysis in JMP.

**How this notebook works**

- Each demo follows the same steps: load and look, run the model, read the output in plain English, check against JMP, say it to a manager.
- JMP is the ground truth. If Python and JMP disagree, Python (or the code you were given) is the one to question.
- GitHub Copilot is on. Use it, and check what it writes against your JMP result before you believe it.
- Run cells in order with **Shift+Enter**. The data files live in `../data/`.

---
## Setup

In [ ]:
# Load the tools we need: pandas for tables, matplotlib for charts,
# scipy for correlation, statsmodels for the fitted line
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import statsmodels.api as sm

# Show the data files that came with this repository
print(sorted(p.name for p in Path("../data").iterdir()))

---
## Demo: ad spend and signups

**The business question:** does monthly ad spend move new customer signups, and by how much per £1,000?

### Step 1: load and look

In [ ]:
# Read the Excel file, then give the columns short names.
# The originals carry units and symbols, which are awkward to type in code.
ad = pd.read_excel("../data/01_Demo_SaaS_AdSpend.xlsx")
ad = ad.rename(columns={"Ad Spend (£1000)": "spend", "New_Customer_Signups": "signups"})
ad.head()

In [ ]:
# 36 months of data. Check the ranges before modelling anything:
# a model should only be trusted inside the range it has seen
ad.describe().round(1)

### Step 2: plot first

Always look at the picture before any number.

In [ ]:
# One dot per month: spend across, signups up
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ad["spend"], ad["signups"])
ax.set_xlabel("Ad spend (£1,000 a month)")
ax.set_ylabel("New customer signups")
plt.show()

### Step 3: correlation

Pearson's r measures the strength of a straight-line pattern. Spearman's rho ranks the values first, so it is a check that the pattern is steady and not driven by a few extreme points.

In [ ]:
# r runs from -1 to +1: the sign is the direction, the size is the strength
r, p_r = stats.pearsonr(ad["spend"], ad["signups"])
rho, p_rho = stats.spearmanr(ad["spend"], ad["signups"])
print(f"Pearson r    = {r:.3f}")
print(f"Spearman rho = {rho:.3f}")

### Step 4: fit the line

The fitted line is the one that makes the misses as small as possible, counting big misses extra. `statsmodels` wants the result first (`signups`) and the driver second (`spend`).

In [ ]:
# add_constant gives the line an intercept, otherwise it is forced through zero
X = sm.add_constant(ad["spend"])
model = sm.OLS(ad["signups"], X).fit()   # result (y) first, driver (x) second
print(model.summary())

### Step 5: read the output in plain English

| Look for | What it says here |
|---|---|
| `coef` for `spend` | The slope: signups that go with each extra £1,000 of spend |
| `coef` for `const` | The intercept: where the line sits at £0 of spend |
| `[0.025  0.975]` | The 95% interval for each coefficient |
| `P>|t|` for `spend` | The p-value for the slope. Shown as 0.000 when it is below 0.0005 |
| `R-squared` | The share of the variation in signups that lines up with spend |

Pull out the numbers you would quote to a manager.

In [ ]:
# Each number a manager might ask for, with units
intercept = model.params["const"]
slope = model.params["spend"]
low, high = model.conf_int().loc["spend"]      # 95% interval for the slope
miss = np.sqrt(model.mse_resid)                # a typical miss of the line, in signups

print(f"Intercept: {intercept:.2f} signups at £0 spend")
print(f"Slope: {slope:.2f} signups per extra £1,000 (95% interval {low:.2f} to {high:.2f})")
print(f"p-value for the slope: {model.pvalues['spend']:.1e}")
print(f"R-squared: {model.rsquared:.3f}")
print(f"A typical miss: {miss:.1f} signups")

### Step 6: predict, with the right interval

A prediction is an average. The interval for the **average month** answers "what does spend do on average?" The interval for **one new month** also includes month-to-month scatter, so it is wider.

In [ ]:
# Three spend levels, all inside the range of the data (4.7 to 25.6)
levels = [8, 10, 15]
new = sm.add_constant(pd.DataFrame({"spend": levels}), has_constant="add")
frame = model.get_prediction(new).summary_frame(alpha=0.05)

table = pd.DataFrame({
    "spend": levels,
    "predicted": frame["mean"],
    "average month, low": frame["mean_ci_lower"],
    "average month, high": frame["mean_ci_upper"],
    "one new month, low": frame["obs_ci_lower"],
    "one new month, high": frame["obs_ci_upper"],
})
table.round(1)

### Step 7: look at the misses

If the line is a fair summary, the misses (residuals) scatter evenly around zero. A curve or a fan shape means the straight line is hiding something.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(ad["spend"], model.resid)
ax.axhline(0)
ax.set_xlabel("Ad spend (£1,000 a month)")
ax.set_ylabel("Miss (signups)")
plt.show()

### Step 8: check against JMP

Run the same analysis in JMP. The values below are what JMP gives; replace them with the ones on your own screen if they differ, then run the cell.

In [ ]:
# Compare the Python result with the JMP result, number by number
jmp = {"Intercept": 14.05, "Slope": 10.05, "R-squared": 0.883}
py = {"Intercept": model.params["const"], "Slope": model.params["spend"], "R-squared": model.rsquared}

for name in jmp:
    ok = np.isclose(jmp[name], py[name], atol=0.01)
    print(f"{name:10}  JMP {jmp[name]:8.3f}   Python {py[name]:8.3f}   match: {ok}")

### Try this

Point the same code at a different file. Change three things (the file name, the two column names and the axis labels), re-run Steps 1 to 6, and compare with your JMP result for that file:

- `../data/02_GuidedPractice_Supermarket.xlsx`: `Discount_Percentage` and `Sales_Uplift_£`
- `../data/03_GuidedPractice_PriceUnits.xlsx`: `Product_Price_£` and `Units_Sold_Monthly`

Or write a short prompt to Copilot asking it to do this for you, and check what it gives you against JMP.

---
## Exercise: fuel efficiency and weight

**The business question:** a car maker wants to know how much highway fuel economy is lost for each extra 1,000 lb of vehicle weight. The file has 200 models from 2011.

### Step 1: do it in JMP first

Open `fuelefficiency2011.xlsx` in JMP. Fit a straight line of `MPG Highway` (y) on `Curb Weight` (x). Write down the intercept, the slope and R-squared.

### Step 2: load the data

In [ ]:
# Load the fuel file and give the columns short names
fuel = pd.read_excel("../data/fuelefficiency2011.xlsx")
fuel = fuel.rename(columns={"MPG Highway": "mpg", "Curb Weight": "weight"})
fuel[["mpg", "weight"]].describe().round(1)

### Step 3: ask Copilot, then check

Write a short prompt that tells Copilot four things: the data frame (`fuel`), the result (`mpg`), the driver (`weight`), and what you want printed (intercept, slope, R-squared and the 95% interval for the slope). Paste or write the code it gives you in the cell below and run it.

In [ ]:
# Your Copilot code goes here


### Step 4: check against JMP

Type your JMP numbers into the cell below (leave a value as `None` if you did not record it), then run it. Check three things as well: does the slope have the sign you expect, are the units sensible, and does the answer match JMP?

In [ ]:
# Type your JMP numbers here, then compare them with the Python output above
jmp_fuel = {"Intercept": None, "Slope": None, "R-squared": None}
print(jmp_fuel)

### Commentary

In your own words: how many miles per gallon does the line say a car loses for every extra 1,000 lb, and does the straight line look like a fair summary of the 200 models?

*Your commentary here*

---
## Exercise: check an assistant's code

An assistant was asked: *fit a line showing how many extra signups each extra £1,000 of ad spend brings.* It wrote the code below. It runs without an error.

Compare its numbers with your JMP result from the demo. Do they agree? If not, find the problem and fix it.

In [ ]:
# Assistant's answer: fit a straight line and report the slope and R-squared
x = ad["signups"]
y = ad["spend"]
assistant_model = sm.OLS(y, sm.add_constant(x)).fit()
print(f"Slope: {assistant_model.params['signups']:.4f}")
print(f"R-squared: {assistant_model.rsquared:.3f}")

In [ ]:
# Your fix goes here


### Commentary

What went wrong, and why did R-squared not give it away?

*Your commentary here*

---
## Say it to a manager

Two sentences, in business language, using your own output: how many extra signups per extra £1,000 of ad spend, how sure you are, and one limit on what the data can tell us.

*Your two sentences here*

---
## Stretch

A blank space for quick work. Use Copilot to take the analysis further, then check the numbers against JMP. Some ideas: plot the line with its interval band, fit `housingprices.xlsx` (`Square Feet` against `Price`) and find the house that pulls the line most, or fit `Miles to Resort` against `Price` and compare the slopes.